# Speech Command Recognition Demo

This notebook loads the trained model from the repository and provides a Gradio interface for uploading a WAV file and getting the predicted command with confidence scores.


In [ ]:
# Install dependencies
%pip install -q gradio tensorflow librosa numpy


In [ ]:
import tensorflow as tf
import numpy as np
import librosa
import json
import gradio as gr

# Load model and label mapping (download from GitHub)
!wget -q https://github.com/nagavishwa-spidey/Speech-Command-Recognition/raw/main/speech_command_model.keras -O speech_command_model.keras
!wget -q https://github.com/nagavishwa-spidey/Speech-Command-Recognition/raw/main/labels.json -O labels.json
model = tf.keras.models.load_model('speech_command_model.keras')
with open('labels.json') as f:
    label_map = json.load(f)

def preprocess_wav(file_path, sr=16000, n_mels=40, duration=1.0):
    y, _ = librosa.load(file_path, sr=sr, duration=duration)
    if len(y) < sr * duration:
        y = np.pad(y, (0, int(sr * duration) - len(y)))
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    log_mel = librosa.power_to_db(mel)
    return np.expand_dims(log_mel.T, axis=(0, -1))  # shape (1, time, n_mels, 1)

def predict(audio_file):
    x = preprocess_wav(audio_file)
    probs = model.predict(x)[0]
    pred_idx = int(np.argmax(probs))
    pred_label = label_map[str(pred_idx)]
    confidence = float(probs[pred_idx])
    return {
        'prediction': f'{pred_label} ({confidence*100:.2f}% confidence)',
        'probabilities': {label_map[str(i)]: float(p) for i, p in enumerate(probs)}
    }

demo = gr.Interface(fn=predict,
                    inputs=gr.Audio(source='upload', type='filepath'),
                    outputs=[gr.Textbox(label='Result'), gr.JSON(label='Class probabilities')],
                    title='Speech Command Recognition Demo',
                    description='Upload a 1‑second WAV file containing one of the keywords (yes, no, up, down, left, right, on, off, stop, go).')

demo.launch(share=False)
